# 03 - Tactics: definitions and inspection

This notebook is the **specification** of the tactic detectors in `src/tactics.py` and a way to **check** them:
for every tactic you get (1) the definition exactly as implemented, (2) its known limits, and (3) hand-made positions with the *expected* answer. Run the cells - every case prints `PASS`/`FAIL` and draws the board (blue arrow = the move, green = what the detector found, red = opponent reply in a line).

**How a detector is used.** A detector answers one question: *"does playing `move` in `board` realise tactic X?"*
- **missed tactic:** `detect(position, best_move)` - the player had a tactic and did not play it,
- **allowed tactic:** `detect(position after the mistake, opponent's best reply)` - the player's mistake gave the opponent a tactic.

So the *type* of the tactic comes from the detector, *for the player or the opponent* from which of the two calls fired, and the *length* will come from the engine line (not done yet).

**Material logic used everywhere.** Piece values P=1, N=3, B=3, R=5, Q=9. "Wins material" means a *static exchange evaluation* (SEE): both sides keep capturing on the square with their cheapest attacker and may stop whenever continuing loses material. Pinned pieces still count as attackers/defenders (simplification).

> Everything below was written without being able to run it against real positions, so the PASS/FAIL tables are the first real test. If a case fails (or you disagree with a definition), copy the row and tell me.

In [1]:
import sys
sys.path.append('../src')

import importlib
import chess
import pandas as pd

import tactics as T
import tactics_inspect as TI
importlib.reload(T); importlib.reload(TI)          # pick up edits without restarting the kernel
from tactics_inspect import Case, run_cases, check_see, inspect_move

results = []      # every run_cases table is collected here (summary at the end)

## 0. Primitive: static exchange evaluation (SEE)
`see(board, move)` = material the mover nets if the capture is played out on that square (negative = the capture loses material). Everything else is built on it.

In [2]:
check_see([
    ("rook takes an undefended bishop",                     "4k3/b7/8/8/8/8/8/R3K3 w - - 0 1",          "a1a7", 3),
    ("knight takes a rook defended by a pawn",              "4k3/8/2p5/3r4/8/2N5/8/4K3 w - - 0 1",      "c3d5", 2),
    ("pawn takes pawn, queen backs up (2 attackers vs 1)",   "4k3/8/2p5/3p4/4P3/8/8/3QK3 w - - 0 1",     "e4d5", 1),
    ("knight takes knight defended by a pawn (even trade)", "4k3/8/2p5/3n4/8/2N5/8/4K3 w - - 0 1",      "c3d5", 0),
    ("rook takes a bishop defended by a rook",              "r3k3/b7/8/8/8/8/8/R3K3 w - - 0 1",          "a1a7", -2),
])

5/5 SEE cases passed


,case,move,expected SEE,SEE,result
0,rook takes an undefended bishop,a1a7,3,3,PASS
1,knight takes a rook defended by a pawn,c3d5,2,2,PASS
2,"pawn takes pawn, queen backs up (2 attackers v...",e4d5,1,1,PASS
3,knight takes knight defended by a pawn (even t...,c3d5,0,0,PASS
4,rook takes a bishop defended by a rook,a1a7,-2,-2,PASS


## 1. Capturing an unprotected piece
**Situation.** An enemy piece is not defended; it can be taken for free.

**Implemented criterion.** `move` is a capture of a non-king piece and **no enemy piece attacks the victim's square**.

**Notes / limits.** Pawns count as victims (parameter `min_victim_value`, default 1). A defender that is pinned still counts as a defender, so some free captures are missed. En passant is not handled.

In [3]:
results.append(run_cases([
    Case("rook takes loose bishop",  T.CAPTURE_UNPROTECTED, "4k3/b7/8/8/8/8/8/R3K3 w - - 0 1",  "a1a7", True),
    Case("same bishop, now defended by a rook", T.CAPTURE_UNPROTECTED, "r3k3/b7/8/8/8/8/8/R3K3 w - - 0 1", "a1a7", False),
]))

2/2 cases passed


## 2. Defending a piece that would otherwise be captured
**Situation.** One of our pieces is attacked and would be lost; another piece can protect it.

**Implemented criterion.** Before the move one of our pieces X (not the king) is *threatened*: the opponent could capture it and win material (SEE > 0). Then **all** of: the move is not a capture; X does not move; after the move X is no longer threatened; the moved piece now defends X and did not defend it before.

**Notes / limits.** Blocking the attack, counter-attacking or capturing the attacker are *not* "defending" here. The book's "has no safe squares to move to" is not checked.

In [4]:
FEN = "4k3/8/1b6/8/3N4/8/8/2B1K3 w - - 0 1"      # black bishop b6 attacks the loose knight d4
results.append(run_cases([
    Case("bishop c1-e3 defends the knight", T.DEFEND_THREATENED, FEN, "c1e3", True),
    Case("the knight just moves away",      T.DEFEND_THREATENED, FEN, "d4e2", False, note="moving away is not defending"),
    Case("irrelevant king move",            T.DEFEND_THREATENED, FEN, "e1e2", False),
]))

3/3 cases passed


## 3. Favourable exchange
**Situation.** We can capture a *more valuable* enemy piece even though our capturing piece can be recaptured.

**Implemented criterion.** `move` is a capture; victim value **>** capturing piece value; the victim is defended (so a recapture is possible); and SEE > 0.

In [5]:
results.append(run_cases([
    Case("knight takes rook defended by a pawn", T.FAVOURABLE_EXCHANGE, "4k3/8/2p5/3r4/8/2N5/8/4K3 w - - 0 1", "c3d5", True),
    Case("knight takes knight defended by a pawn", T.FAVOURABLE_EXCHANGE, "4k3/8/2p5/3n4/8/2N5/8/4K3 w - - 0 1", "c3d5", False,
         note="even trade - victim is not worth more"),
]))

2/2 cases passed


## 4. Capturing a piece that is attacked more often than defended
**Situation.** An enemy piece is attacked by more pieces than defend it.

**Implemented criterion.** `move` is a capture; victim value **<=** capturing piece value; our attackers of the square > their defenders (counted with `board.attackers`); SEE > 0; and the capturing piece is still protected after the capture.

**Note.** Together with 1 and 3 this splits all "good captures" by (victim defended?, victim worth more than the capturer?), so a capture normally gets exactly one of the three labels.

In [6]:
results.append(run_cases([
    Case("pawn takes pawn, queen behind (2 vs 1)", T.CAPTURE_ATTACKED_MORE, "4k3/8/2p5/3p4/4P3/8/8/3QK3 w - - 0 1", "e4d5", True),
    Case("same without the queen (1 vs 1)",         T.CAPTURE_ATTACKED_MORE, "4k3/8/2p5/3p4/4P3/8/8/4K3 w - - 0 1",   "e4d5", False),
]))

2/2 cases passed


## 5. Double attack (fork)
**Situation.** One piece attacks two targets at once; the opponent can only parry one.

**Implemented criterion.** After the move, the **moved piece** (not the king) attacks at least two *targets*, where a target is
- the enemy king (this piece gives check), or
- an enemy piece the moved piece could capture with a good capture (labels 1, 3 or 4 above).

The forking piece must be **safe**: not attacked, or defended and attacked only by more valuable pieces.

**Not implemented.** Targets that are "important squares"; forks created by a *discovered* attack (see 9); the opponent's actual reply (the static version ignores it).

In [7]:
FEN = "r3k3/8/8/3N4/8/8/8/4K3 w - - 0 1"
results.append(run_cases([
    Case("Nc7+ forks king and rook", T.DOUBLE_ATTACK, FEN, "d5c7", True),
    Case("Nb6 attacks only the rook", T.DOUBLE_ATTACK, FEN, "d5b6", False),
]))

2/2 cases passed


## 6. Pin
**Situation.** A piece cannot (safely) leave its line because a more valuable piece / the king stands behind it.

**Implemented criterion.** After the move the moved piece is a **bishop, rook or queen**, is **safe**, and along one of its lines sees an enemy piece F (not the king) with another enemy piece B *directly behind* it, where B is the enemy king (**absolute** pin) or is worth **more** than F (**relative** pin) *and* the pin is real: if F left the line, the pinner could capture B and **win material** (static exchange evaluation > 0 - i.e. B is undefended or worth more than the pinner). A queen "pinning" a pawn to a knight that is defended by a pawn is therefore **not** a pin. If the opponent's best reply is supplied (`opp_reply`), the pin only counts when that reply does not move F off the line.

**The pin must be NEW:** if the same pair (F, B) was already pinned by one of our pieces before the move (e.g. the pinning queen just retreats along the same line), it is not a pin tactic.

**Not implemented.** Pins to an "important square"; skewers (front piece more valuable than the back piece).

In [8]:
results.append(run_cases([
    Case("Bb5 pins the knight to the king",       T.PIN, "4k3/8/2n5/8/8/8/8/4KB2 w - - 0 1",   "f1b5", True,  note="absolute pin"),
    Case("Bc4 attacks nothing in line",           T.PIN, "4k3/8/2n5/8/8/8/8/4KB2 w - - 0 1",   "f1c4", False),
    Case("Bb5 pins the knight to the queen",      T.PIN, "4k3/3q4/2n5/8/8/8/8/4KB2 w - - 0 1", "f1b5", True,  note="relative pin"),
    Case("behind the knight is only a pawn",      T.PIN, "4k3/3p4/2n5/8/8/8/8/4KB2 w - - 0 1", "f1b5", False, note="back piece is worth less"),
    Case("Qa8 keeps the OLD pin on Ba5/Qa4", T.PIN, "4kb1r/3n1ppp/q4n2/B7/Q1Np4/7P/P4PP1/1R4K1 b k - 0 24", "a6a8", False,
         note="real position (review row 3201): the pin already existed"),
    Case("queen vs pawn, knight behind is pawn-defended", T.PIN, "r2qkb1r/3n1pp1/p1nPp2p/1p5b/3P1B2/1BN2N1P/PP3PP1/R2QK2R b KQkq - 1 14", "d8f6", False,
         note="real position (review row 4015): Qxc3 would lose the queen for a knight"),
]))

6/6 cases passed


## 7. Elimination of the defence
**Situation.** An enemy piece T is defended; we remove / distract / chase away the defender and then win T.

**Needs the engine's line** `pv = [our move, opponent reply, our second move]`.

**Implemented criterion.** The second move captures an enemy piece T with a good capture; T was **not** a good capture before the first move (it was adequately defended); and after the first two moves T has **fewer defenders**. Sub-kind, checked in this order:
1. `capture_defender` - the first move captures a defender of T,
2. `lure` - the opponent's reply is a defender of T capturing our first piece (sacrifice / exchange),
3. `chase` - the first move attacks a defender of T and the reply moves it away,
4. `other`.

In [9]:
results.append(run_cases([
    Case("Bxf6 removes the defender of Rd7", T.ELIMINATION, "6k1/3r4/5n2/8/7B/8/8/3RK3 w - - 0 1", "h4f6", True,
         pv=["h4f6", "g8f8", "d1d7"], note="capture_defender"),
    Case("e5 chases the defender away",      T.ELIMINATION, "6k1/3r4/5n2/8/4P3/8/8/3RK3 w - - 0 1", "e4e5", True,
         pv=["e4e5", "f6h5", "d1d7"], note="chase"),
    Case("Nxh7 lures the defender away",     T.ELIMINATION, "1k6/3r3p/5n2/6N1/8/8/8/3RK3 w - - 0 1", "g5h7", True,
         pv=["g5h7", "f6h7", "d1d7"], note="lure"),
    Case("line where the 2nd move is no capture", T.ELIMINATION, "6k1/3r4/5n2/8/7B/8/8/3RK3 w - - 0 1", "h4f6", False,
         pv=["h4f6", "g8f8", "e1e2"]),
]))

4/4 cases passed


## 8. Forced mate
**Situation.** A sequence of moves forces checkmate.

**Implemented criterion.** With the engine score of the best move (`best_cp`, as stored in our CSVs: a mate in *n* is `10000 - n`): mate in at most `max_mate` moves (default 5). Without a score: the move itself gives checkmate (mate in 1, checked on the board).

**Reading the cases below.** The first two cases look at the real board. The three `SCORE ONLY` cases do **not**: they type in an engine score (`10000 - n` = mate in *n*) and only test that the number is converted and compared with `max_mate` correctly - the position drawn next to them is a placeholder. With real data the score comes from the `best_cp` column of the CSV (section 11).

In [10]:
BACK_RANK = "6k1/5ppp/8/8/8/8/8/3R2K1 w - - 0 1"
START = chess.STARTING_FEN
results.append(run_cases([
    Case("Rd8# (back-rank mate)",           T.FORCED_MATE, BACK_RANK, "d1d8", True),
    Case("Rd2 is not mate",                 T.FORCED_MATE, BACK_RANK, "d1d2", False),
    Case("SCORE ONLY: typed-in 9997 is read as mate in 3", T.FORCED_MATE, START, "e2e4", True,  ctx={"best_cp": 9997}),
    Case("SCORE ONLY: typed-in 250 is just +2.5, no mate", T.FORCED_MATE, START, "e2e4", False, ctx={"best_cp": 250}),
    Case("SCORE ONLY: typed-in 9993 = mate in 7, above max_mate=5", T.FORCED_MATE, START, "e2e4", False, ctx={"best_cp": 9993}),
]))

5/5 cases passed


## 9. Discovered attack
**Situation.** A piece M moves away and uncovers an attack by a slider B (bishop/rook/queen) standing behind it.

**Threat types** (what a piece of ours threatens after the move):
- `king` - it gives check;
- `material` - it attacks an enemy piece it could capture with a good capture (labels 1, 3, 4);
- `square` - it attacks an **empty important square** X: if the piece went to X (our turn) it would give **checkmate** (*mating square*) or **fork** two targets (*fork square*). Squares are defined through king/material one level deeper, so there is no endless recursion; deeper definitions (e.g. "a square from which we force material") can be added the same way.

**Implemented criterion.** B (not M) attacks after the move a square it did not attack before, B is **safe**, and the newly attacked squares hold a threat of B (king > material > square). 
**Classification `kind`** = strongest threat of B + strongest threat of the moved piece M, written in the order king < material < square: `king+king` (double check), `king+material`, `king+square`, `material+material`, `material+square`, `square+square`; and `king+none`, `material+none`, `square+none` when only B threatens something (M's move is "quiet").

**Not implemented.** Squares that need a deeper plan (forced material win in a few moves); discovered attacks that need the opponent's reply to be checked.

In [11]:
FEN = "3qk3/8/8/8/3N4/8/8/3RK3 w - - 0 1"       # Nd4 blocks the d-file between Rd1 and the queen on d8
FEN_K = "4k3/1q6/8/8/4N3/8/8/4R1K1 w - - 0 1"   # Ne4 blocks the e-file between Re1 and the king
FEN_MM = "3qk3/r7/8/8/3N4/8/8/3RK3 w - - 0 1"   # as FEN, plus a black rook on a7
FEN_SQ = "6k1/5ppp/8/8/4N3/8/8/4R1K1 w - - 0 1"  # back-rank: e8 would be mate
results.append(run_cases([
    Case("Nf5 uncovers Rd1 on the queen",         T.DISCOVERED_ATTACK, FEN, "d4f5", True, expect_kind="material+none"),
    Case("Nd5 stays on the file (still blocks)",  T.DISCOVERED_ATTACK, FEN, "d4d5", False),
    Case("Nc5+ discovered check, knight hits Qb7", T.DISCOVERED_ATTACK, FEN_K, "e4c5", True, expect_kind="king+material"),
    Case("Nd6++ double check",                    T.DISCOVERED_ATTACK, FEN_K, "e4d6", True, expect_kind="king+king"),
    Case("Nb5 uncovers Rd1 on Qd8, hits Ra7",    T.DISCOVERED_ATTACK, FEN_MM, "d4b5", True, expect_kind="material+material"),
    Case("Nc3 uncovers Re1: e8 is a mating square", T.DISCOVERED_ATTACK, FEN_SQ, "e4c3", True, expect_kind="square+none"),
    Case("Nf8+ uncovers Qc2 on the king (double check)", T.DISCOVERED_ATTACK, "r1q1r3/pp1b1Bpk/1n1p1nNp/8/8/7P/PPQN1PP1/3RR1K1 w - - 3 21", "g6f8", True,
         expect_kind="king+king", note="real position (review row 2374)"),
]))


6/7 cases passed


## 10. Defending against mate
**Situation.** The opponent threatens mate and only one move prevents it; every other move allows mate.

**Implemented criterion.**
- With engine scores (`best_cp`, `second_cp`): the **second-best** move is mated in at most `max_mate` moves while the best move is not (so every alternative loses to mate as well).
- Without scores (mate in **one** only): `move` is the only legal move after which the opponent has no mate in one.

**Reading the cases below.** The `axb3` / `a3` / `Kb1` cases look at the real board. The `SCORE ONLY` cases only test the score arithmetic (placeholder board).

In [12]:
FEN = "4r2k/8/8/8/8/1p6/PP6/K7 w - - 0 1"       # Black threatens ...Re1#; only axb3 makes room on a2
results.append(run_cases([
    Case("axb3 is the only defence",        T.DEFEND_AGAINST_MATE, FEN, "a2b3", True),
    Case("a3 still gets mated by Re1#",     T.DEFEND_AGAINST_MATE, FEN, "a2a3", False),
    Case("Kb1 still gets mated by Re1#",    T.DEFEND_AGAINST_MATE, FEN, "a1b1", False),
    Case("SCORE ONLY: 2nd best typed in as mated in 2", T.DEFEND_AGAINST_MATE, START, "e2e4", True,  ctx={"best_cp": 0, "second_cp": -9998}),
    Case("SCORE ONLY: 2nd best typed in as -1.5, no mate", T.DEFEND_AGAINST_MATE, START, "e2e4", False, ctx={"best_cp": 0, "second_cp": -150}),
    Case("SCORE ONLY: best move is mated too (-9997)", T.DEFEND_AGAINST_MATE, START, "e2e4", False, ctx={"best_cp": -9997, "second_cp": -9998}),
]))

6/6 cases passed


## Summary of all hand-made cases

In [13]:
summary = pd.concat(results, ignore_index=True)
print(f"{int((summary['result'] == 'PASS').sum())}/{len(summary)} passed")
summary[summary['result'] != 'PASS']       # empty = everything passed

38/39 passed


,case,tactic,expected,detected,result,details,also detected
27,Nd5 stays on the file (still blocks),discovered_attack,False,None,BAD CASE (illegal move),,


## 10b. Prevent a threat (prophylaxis)
**Situation.** The opponent is threatening a tactic (e.g. a discovered check or a fork) and our move takes that threat away - without any piece being attacked yet. This is the "defending an important square" idea.

**Implemented criterion.** Imagine we **passed**: the opponent could play a set T0 of tactical moves - every legal reply is scanned with `forced_mate` (mate in 1), `discovered_attack` and `double_attack`. After our move the same scan gives T1. The move **prevents a threat** when some (reply, tactic) pair from T0 is no longer in T1.

**Not counted.** Moves that give check (the reply is forced anyway); positions where we are in check; opponent pins and plain captures (captures belong to `defend_threatened_piece`). No pruning: all legal replies are scanned (results are cached per position).

In [14]:
FEN_P = "4k3/1q6/8/8/4N3/8/8/4R1K1 b - - 0 1"   # white threatens Nc5+/Nd6+ (discovered check from Re1)
results.append(run_cases([
    Case("Kf8 leaves the e-file: threats disappear", T.PREVENT_THREAT, FEN_P, "e8f8", True),
    Case("Qb5 changes nothing (checks still work)",  T.PREVENT_THREAT, FEN_P, "b7b5", False),
]))

2/2 cases passed


## 11. Try it on real positions
Draws the best move of real mistakes from `critical_positions.csv` and lists every tactic the detectors recognise (`best_cp` / `second_cp` are used for the mate tactics). Change `random_state` for another sample, or call `inspect_move(fen, uci)` on any position. This is also the quickest way to see **which share of mistakes is classified at all**.

In [15]:
dfn = pd.read_csv('../data/critical_positions.csv')
mistakes = dfn[dfn['is_critical'] & ~dfn['found_best']]
sample = mistakes.sample(8, random_state=1)
for r in sample.itertuples():
    inspect_move(r.fen, r.best_move, best_cp=r.best_cp, second_cp=r.second_cp)

In [16]:
# share of critical mistakes where at least one tactic is recognised for the BEST move (static detectors only;
# elimination needs a pv, so it is not included here)
def labels_for(row):
    board = chess.Board(row.fen)
    move = chess.Move.from_uci(row.best_move)
    return [d.tactic for d in T.detect_all(board, move, best_cp=row.best_cp, second_cp=row.second_cp)]

check = mistakes.sample(min(300, len(mistakes)), random_state=0)
check = check.assign(tactics=[labels_for(r) for r in check.itertuples()])
print(f"{(check['tactics'].str.len() > 0).mean():.1%} of {len(check)} sampled mistakes have at least one recognised tactic")
check['tactics'].explode().value_counts()

57.7% of 300 sampled mistakes have at least one recognised tactic


tactics
capture_unprotected                    67
prevent_threat                         49
double_attack                          31
capture_attacked_more_than_defended    18
pin                                    14
defend_threatened_piece                13
discovered_attack                      12
favourable_exchange                     8
forced_mate                             4
defend_against_mate                     3
Name: count, dtype: int64

## Open questions
1. **Pawns as victims** - should "capturing an unprotected piece" include pawns? (now: yes)
2. **"Attacked more than defended"** - I judge material with SEE instead of only counting attackers vs defenders (a rook attacked by a pawn is lost even if defended twice). OK?
3. **Missed vs allowed** - I assume: *missed* = detectors on (position, best move); *allowed* = detectors on (position after the mistake, opponent's best reply). Is that the intended meaning of "tactic for the player or for the opponent"? What if both apply?
4. **Mate tactics** - what is "a few moves"? (now: `max_mate = 5`)
5. **Elimination of the defence** needs the engine's line (3 plies) - I would re-run Stockfish on the critical positions only to get it.